# Study 924 — First Cut

**When the Fed starts cutting, is that the moment to buy duration?**

The folk trade: the FOMC delivers the **first cut of an easing cycle**, you buy long
Treasuries, and you ride a multi-quarter bond rally. We tested it the only way it can be
tested — a hardcoded list of cycle-start cuts, buy **TLT** at the **close of the session
after** the announcement, hold 1 / 3 / 6 / 12 months, score it **excess of T-bills (BIL)**,
5 bps one-way each way.

The tape is TLT vs BIL daily **total-return** closes, 2007-05-30 → 2026-06-30
(4,802 days), as-of 2026-06-30.

**And the number that governs everything below: N = 4.**

*Real numbers are the frozen headline from `docs/results.md` (Fingerprint `ff993b355f57`);
the live cells run the offline synthetic control only.*


## 1. Four trades. That is the entire dataset.

Since 2001 the Fed has begun an easing cycle five times by any reasonable reading: January 2001, September 2007, July 2019, the emergency cut of March 2020, and September 2024. The first of those pre-dates the long-Treasury ETF, so **four** trades is what the tape allows. Here they are, in full — this is not a summary of the evidence, it *is* the evidence.

In [1]:
events = [('2007-09-18', '2007-09-19', '2008-09-18', 14.37, 2.78, 11.5), ('2019-07-31', '2019-08-01', '2020-07-31', 28.58, 1.11, 27.37), ('2020-03-03', '2020-03-04', '2021-03-03', -8.51, 0.05, -8.67), ('2024-09-18', '2024-09-19', '2025-09-18', -6.2, 4.36, -10.66)]
print(f"{'first cut':12s} {'entry':12s} {'exit':12s} {'TLT':>8s} {'BIL':>7s} {'excess net':>11s}")
for ev, entry, ex, tlt, bil, net in events:
    print(f'{ev:12s} {entry:12s} {ex:12s} {tlt:+8.2f} {bil:+7.2f} {net:+11.2f}')
mean = sum(e[5] for e in events) / len(events)
print(f'\nN = {len(events)}   mean excess net = {mean:+.2f}%   hit rate = {sum(1 for e in events if e[5] > 0)}/{len(events)}')
print('2001-01-03 is unmeasurable: TLT only lists from 2002-07-30.')

first cut    entry        exit              TLT     BIL  excess net
2007-09-18   2007-09-19   2008-09-18     +14.37   +2.78      +11.50
2019-07-31   2019-08-01   2020-07-31     +28.58   +1.11      +27.37
2020-03-03   2020-03-04   2021-03-03      -8.51   +0.05       -8.67
2024-09-18   2024-09-19   2025-09-18      -6.20   +4.36      -10.66

N = 4   mean excess net = +4.88%   hit rate = 2/4
2001-01-03 is unmeasurable: TLT only lists from 2002-07-30.


## 2. The average is one trade wide

The mean looks respectable: **+4.88%** over twelve months, above cash, after costs. But two of the four lost money, and the whole positive average rests on the 2019 window — which happens to run from August 2019 to July 2020, i.e. straight through the pandemic flight-to-quality. Drop it and the other three average **-2.61%**.

The 2024 cut is the honest counter-example: the Fed cut, and long bonds fell **−6.2%** over the next year while T-bills paid **+4.4%** — a **−10.7%** excess. Cutting the front end does not oblige the long end to follow.

## 3. The control that quietly demolishes the premise

If the *first* cut is special, buying it should beat buying cuts in general. It does not. Over twelve months, buying duration after **any** of the 18 measurable cuts paid **more** than buying after the four hand-picked first ones.

In [2]:
first = {1: (3.18, 3.08, 4.61, 0.91, 0.75, 40.13, 2.22, 1.04), 3: (0.42, 0.32, 3.71, 0.12, 0.75, 3.44, 2.86, 0.18), 6: (3.79, 3.69, 7.99, 0.9, 0.75, 8.67, 2.22, 0.81), 12: (4.98, 4.88, 1.51, 0.55, 0.5, 2.62, 2.95, 0.35)}
allc = {1: (0.96, 0.86, 0.8, 0.44, 0.78), 3: (0.54, 0.44, 0.31, 0.5, 0.6), 6: (1.3, 1.2, 0.64, 0.61, 0.65), 12: (5.6, 5.5, 1.78, 0.72, 0.22)}
print(f"{'horizon':>8s} {'FIRST cuts (N=4)':>20s} {'ALL cuts (N=18)':>20s}")
for h in (1, 3, 6, 12):
    print(f'{h:>6d}m {first[h][1]:>+15.2f}%      {allc[h][1]:>+15.2f}%')
print('\nThe hand-picked label buys you nothing and costs you 14 observations.')

 horizon     FIRST cuts (N=4)      ALL cuts (N=18)
     1m           +3.08%                +0.86%
     3m           +0.32%                +0.44%
     6m           +3.69%                +1.20%
    12m           +4.88%                +5.50%

The hand-picked label buys you nothing and costs you 14 observations.


> 🔬 **For the quants** — the all-cuts leg is not a clean control either: adjacent cuts share overlapping 12-month windows (53 of 153 pairs overlap), so those 18 events collapse onto just **3 macro episodes** — 2007-08, 2019-20 and 2024-25 — and its nominal *t* = +1.78 is badly oversized. Read it as a mean, not as a test. The point is comparative, not absolute: whatever is in the data is a *"the Fed is easing"* effect, not a *"this is the first one"* effect.

## 4. What a random day would have bought you

The right question for a four-event mean is not "is it positive?" but "how often would four random dates have done as well?" Owning duration paid something over this sample anyway, so a random 12-month start already earns **+2.61%** above cash. The first-cut premium above *that* is about two points, and 2,000 random draws produce a mean at least that good **36% of the time**. Only the one-month window is even suggestive (*p* = 0.086), and we looked at four horizons.

## 5. Why we cannot simply say "it's false"

Here is the uncomfortable part, and the live cell below shows it. We built a synthetic world with a **real** post-cut rally planted in it — a genuine **+9%** six-month effect — and ran the identical five-event test on it, world after world. It found the effect in only **5 of 12** worlds.

So a five-event study *cannot* reliably detect an effect big enough to change your life. Pool the worlds together and the machinery recovers the planted number almost exactly — the harness is fine. The sample is not.

In [3]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from first_cut import data, strategy as st

def five_event_ts(signal_strength):
    ts, pooled = [], []
    for prices, events, _ in data.synthetic_panel(n_worlds=12, signal_strength=signal_strength):
        tbl = st.event_table(prices['duration'], prices['cash'], events, 6, 5.0)
        ts.append(st.one_sample_t(tbl['excess_net_pct'].to_numpy()))
        pooled.extend(tbl['excess_net_pct'].tolist())
    return np.array(ts), np.array(pooled)

for tag, ss in [('planted +9% effect', 1.0), ('null, no effect   ', 0.0)]:
    ts, pooled = five_event_ts(ss)
    print(f'{tag}: pooled mean {pooled.mean():+5.2f}% (t={st.one_sample_t(pooled):+5.2f}, '
          f'N={len(pooled)})  |  five-event t clears 2 in {(np.abs(ts) > 2).sum()}/12 worlds')
print('\nSYNTHETIC DATA — a machinery check, not the real tape.')

planted +9% effect: pooled mean +8.55% (t=+6.93, N=60)  |  five-event t clears 2 in 5/12 worlds


null, no effect   : pooled mean -0.83% (t=-0.74, N=60)  |  five-event t clears 2 in 1/12 worlds

SYNTHETIC DATA — a machinery check, not the real tape.


## Verdict

- **Signal — None.** Every horizon's point estimate is positive and not one clears *t* = 1.1 (12-month: **+4.88%**, *t* = +0.55, N = 4). A random start date already buys +2.61%, so the placebo *p* is **0.365**. Buying *any* cut paid more than buying the first one. The two pre-2020 events made **+19.4%**, the two since made **-9.7%**.
- **Tradability — Mirage.** The conditional *strategy* — long TLT inside the windows, cash the other 81.3% of days — earns **+0.47%/y** excess of cash across the whole sample, CI [-2.05%, +3.02%]. Measured only over the days it is actually invested it makes +2.62%/y — still *less* than the +2.89%/y you get from just owning TLT and going for a walk. It loses to buy-and-hold on either reading.
- **The honest caveat.** This study cannot prove the trade is worthless; four events can barely prove anything. What it can say is that nothing survives contact with a control, and that anyone sizing this position is making a four-sample bet on a story.